In [ ]:
using Pkg
toktox_path = "/Users/adrianaghiozzi/Code/Sources/dev/TokToxStability"   # adjust to your clone
Pkg.activate(toktox_path)
Pkg.instantiate()

In [ ]:
venv_python = joinpath(toktox_path, ".venv", "bin", "python")
if !isfile(venv_python)
    run(`$(joinpath(toktox_path, "setup_venv.sh"))`)
end

In [ ]:
ENV["JULIA_CONDAPKG_BACKEND"] = "Null"
ENV["JULIA_PYTHONCALL_EXE"] = venv_python

In [ ]:
using Revise
using TokToxStability
using Plots
using DataFrames
using PythonCall

In [ ]:
# only call this once per kernel session
myOFT = TokToxStability.make_oft_env()

In [ ]:
# toroidal mode number can be changed here, as can the wall setting (default is no-wall)
gpec_settings = TokToxStability.load_default_settings_kink(1)


In [ ]:
# Create initial guess profiles for TokaMakerTORAX
psi_sample = collect(range(0.0, 1.0, length=200))

ne_core = 5.0e19   # m^-3
ne_edge = 7.0e18   # m^-3 

te_core = 5.0       # keV
te_edge = 0.1      # keV

ti_core = 5.0       # keV
ti_edge = 0.1      # keV

psi_ped    = 0.90                     
width      = 0.15     
core_slope = -2.0     
blend      = 0.05

ne_init = TokToxStability.mtanh_pedestal_profile(psi_sample, ne_core, ne_edge, core_slope, psi_ped, width, blend)
te_init = TokToxStability.mtanh_pedestal_profile(psi_sample, te_core, te_edge, core_slope, psi_ped, width, blend)
ti_init = TokToxStability.mtanh_pedestal_profile(psi_sample, ti_core, ti_edge, core_slope, psi_ped, width, blend)

In [ ]:
# Set up TokaMakerTORAX initial input
d3d_case_dir = joinpath(pkgdir(TokToxStability), "cases", "D3D")

tmtx_input = TokToxStability.TMTXInput(
    seed_geqdsk  = joinpath(d3d_case_dir, "g163303.05000_DIIID_seed.eqdsk"), # sample seed eqdsk from shot 163303
    Ip           = 1.0e6,              
    x_points     = [1.2731 -1.1807],
    psi_sample   = psi_sample,
    ne_init      = ne_init,
    Te_init      = te_init,
    Ti_init      = ti_init,
    ne_edge      = ne_edge,
    Te_edge      = te_edge,
    Ti_edge      = ti_edge,
    pedestal     = Dict(
        "set_pedestal"      => true,
        "model_name"        => "set_T_ped_n_ped",
        "mode"              => "ADAPTIVE_SOURCE",
        "pedestal_profile_form" => "MTANH",
        "rho_norm_ped_top"  => psi_ped,   # NOTE: TORAX's pedestal config uses rho_norm, not psi_norm -- see caveat below
        "T_i_ped"           => ti_core * 0.2,  # rough pedestal-top guess
        "T_e_ped"           => te_core * 0.2,
        "n_e_ped_is_fGW"    => false,
        "n_e_ped"           => ne_core * 0.7,
    ),
    plasma_composition = Dict("main_ion"=>Dict("D"=>1.0), "impurity"=>"C", "Zeff"=>1.5),
    heating = Dict(
        "generic_heat"      => Dict(0.0 => 3.0e6), 
        "generic_heat_loc"  => 0.2,             
        "generic_heat_width"=> 0.20,            
    ),
    fueling = Dict(
        "generic_particle_location" => 0.3,
        "generic_particle_width"    => 0.10,
        "generic_particle_S_total"  => 1.0e21,
    )
)


In [ ]:
# Set up TokaMakerTORAX settings 
tmtx_settings = TokToxStability.TMTXSettings(
    kin_nr = 200,
    last_surface_factor = 0.95,
    max_loop = 1) 

In [ ]:
mygs, profiles, equilibrium = TokToxStability.setup_from_tmtx!(:D3D, tmtx_input, tmtx_settings, myOFT; verbose=true);

In [ ]:
# The gEQDSK files created by TokaMaker at each coarse scan iteration can be saved by setting working_directory
# For a file-less option, set working_directory = nothing, which will save the gEQDSKs to a temporary directory that is deleted after the run
working_directory = nothing

# starting_fraction sets the first scaling of the temperature profile to be applied in the beta limit search
# step sets the spread of the subsequent scalings 
starting_fraction = 1.0 
scan_step = 0.25 

coarse_scan_results = TokToxStability.coarse_scan_until_unstable(mygs, starting_fraction, scan_step, equilibrium, profiles, gpec_settings; working_directory = working_directory);

In [ ]:
# Creates a DataFrame with the results of the scan 
cols = (:fraction, :beta_n, :dW_total, :dW_plasma, :dW_vacuum, :qmax, :msing)
DataFrame([NamedTuple{cols}(r[c] for c in cols) for r in coarse_scan_results])

In [ ]:
# estimate_beta_limit function interpolates the zero crossing of dW_total based on the scanned points
# If the limit wasn't reached, try re-running coarse_scan_until_unstable with a larger starting_fraction
bl = TokToxStability.estimate_beta_limit(coarse_scan_results)


In [ ]:
TokToxStability.plot_beta_scan_summary(coarse_scan_results; beta_limit=bl)


In [ ]:
TokToxStability.plot_profiles(coarse_scan_results)
